# 05 · Lightning — 1×1 GPU

`lightning_trainer.fit(...)`을 driver에서 직접 호출합니다. `Trainer(devices=1, num_nodes=1, strategy='auto')`. TorchDistributor 없이 노트북 프로세스에서 학습합니다.

> 1×N은 [`06-launch_lightning_trainer_1xN.ipynb`](06-launch_lightning_trainer_1xN.ipynb), M×N은 [`07-launch_lightning_trainer_MxN.ipynb`](07-launch_lightning_trainer_MxN.ipynb)에서 다룹니다.

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (1× A10G), Workers 0.

In [ ]:
%run ./00-setup

## 학습 함수 import

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

from lightning_trainer import fit  # driver에서 직접 호출이라 module reference로도 동작

## 1×1 GPU

In [ ]:
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

fit(
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-1x1"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=1,
    num_nodes=1,
    topology="1x1",
    script_dir=SCRIPT_DIR,
)